In [2]:
##code is implementing a user-based collaborative filtering system. 
#First, it computes how similar user 1 is to each of the other users based on their shared movie ratings. 
#Once those similarities are calculated, it uses them to predict which movies user 1 hasn’t seen 
#but might like—based on the weighted ratings of similar users. 
#In the end, it recommends the top movies for user 1.

In [3]:
# user based recommendations
import pprint
from math import sqrt

In [13]:
# load database and construct the  nested dictionary
def loadMovies():

  #Get movie titles
  movies={}
  for line in open('movies.dat'):
    (id, title) = line.split('|')[0:2]
    movies[id] = title

  #load data
  prefs={}
  for line in open('test.dat'):
    (usr, movieid, rating, ts) = line.split('\t')
    prefs.setdefault(usr, {})
    prefs[usr][movies[movieid]]=float(rating)
  return (movies, prefs)

# compute the Euclidean distance between two person's preferences
def sim_distance(prefs, person1, person2):

  #Get the list of shared items
  si = {}
  for it in prefs[person1]:
    if it in prefs[person2]:
      si[it]=1

  #if they have no ratings in common, return 0
  if (len(si) == 0): return 0

  # Add up the squares of all the differences
  euclidean_distance = sqrt(sum([pow(prefs[person1][it] - prefs[person2][it], 2) 
                    for it in si]))

  similarity = 1/(1+euclidean_distance)
  return similarity

# compute the pearson correlation coefficient
def sim_pearson(prefs, person1, person2):
 
  #Get the list of shared items
  si = {}
  for it in prefs[person1]:
    if it in prefs[person2]:
      si[it]=1
  
  print(f"Shared items: {si.keys()}")
  # Find the number of elements
  n = len(si)

  # if they have no ratings in common, return 0
  if n==0:  return 0

  # sum
  sum1 = sum([prefs[person1][it] for it in si])
  sum2 = sum([prefs[person2][it] for it in si])

  # sum of the squares
  sum1Square = sum([pow(prefs[person1][it], 2) for it in si])
  sum2Square = sum([pow(prefs[person2][it], 2) for it in si])

  # sum up the products
  pSum = sum([prefs[person1][it]*prefs[person2][it] for it in si])

  # compute the correlation coefficient
  num = pSum - (sum1*sum2/n)
  denom = sqrt((sum1Square - pow(sum1, 2)/n)*(sum2Square - pow(sum2, 2)/n))

  if denom == 0: return 0

  pearson_correlation = num/denom

  return pearson_correlation

# return the top N most similar person/items
def topMatches(prefs, person, n=5, similarity=sim_pearson):
  scores=[(similarity(prefs, person, other), other)
             for other in prefs if other != person]

  # sort the list so the highest scores appear at the top
  scores.sort()
  scores.reverse()

  return scores[0:n]

# Get recommendations for a person by using a weighted average
# of every other user's ranking
def getRecommendations(prefs, person, similarity=sim_pearson):

  totals={}
  simSums = {}

  for other in prefs:
    # don't compare one to himself
    if other == person:
      continue

    sim = similarity(prefs, person, other)
    print("\n========================");
    print(f"Similarity with {other}: {sim}")
  
    #ignore scores of zero or lower
    if sim < 0: 
      continue

    for item in prefs[other]:
      #only score movies I have not seen before
      if item not in prefs[person] or prefs[person][item]==0:
        totals.setdefault(item, 0)
        
        #weighted similarity score = simiarity * score
        totals[item] += prefs[other][item]*sim

        #sum of similarities
        simSums.setdefault(item, 0)
        simSums[item] += sim

        # Debug: print weighted total and similarity sum
        print(f"Item: {item}, Rating by user {other}: {prefs[other][item]},  Similarity sum: {simSums[item]}, Weighted total: {totals[item]}")
       

  # Create the normalized list
  print("\n")
  for item, total in totals.items():
    print(f"Item: {item}, Total score: {total}, Similarity sum: {simSums[item]}")

  rankings = [(total / simSums[item], item)
            for item, total in totals.items()
            if simSums[item] > 0]
  # Retrn the sorted list
  rankings.sort()
  rankings.reverse()
    
  return rankings


In [14]:
# main function
if __name__ == '__main__':

    movies, prefs = loadMovies()
    print(f"Loaded {len(movies)} movies.")
    pp = pprint.PrettyPrinter(indent=2)
    print ("List of movies are:\n")
    pp.pprint(movies)

Loaded 5 movies.
List of movies are:

{ '1': 'Toy Story (1995)',
  '2': 'GoldenEye (1995)',
  '3': 'Four Rooms (1995)',
  '4': 'Get Shorty (1995)',
  '5': 'Copycat (1995)'}


In [15]:
print ("\nThe preference list is:\n")
pp.pprint(prefs)


The preference list is:

{ '1': {'Get Shorty (1995)': 3.0, 'GoldenEye (1995)': 3.0},
  '2': { 'Four Rooms (1995)': 2.0,
         'Get Shorty (1995)': 4.0,
         'GoldenEye (1995)': 3.0,
         'Toy Story (1995)': 2.0},
  '3': { 'Four Rooms (1995)': 2.0,
         'Get Shorty (1995)': 5.0,
         'GoldenEye (1995)': 4.0},
  '4': { 'Copycat (1995)': 4.0,
         'Four Rooms (1995)': 4.0,
         'Get Shorty (1995)': 3.0},
  '5': { 'Copycat (1995)': 5.0,
         'Get Shorty (1995)': 5.0,
         'GoldenEye (1995)': 3.0,
         'Toy Story (1995)': 3.0}}


In [16]:
recommendations = getRecommendations(prefs, '1', sim_pearson)


Shared items: dict_keys(['GoldenEye (1995)', 'Get Shorty (1995)'])

Similarity with 2: 0
Item: Toy Story (1995), Rating by user 2: 2.0,  Similarity sum: 0, Weighted total: 0.0
Item: Four Rooms (1995), Rating by user 2: 2.0,  Similarity sum: 0, Weighted total: 0.0
Shared items: dict_keys(['GoldenEye (1995)', 'Get Shorty (1995)'])

Similarity with 3: 0
Item: Four Rooms (1995), Rating by user 3: 2.0,  Similarity sum: 0, Weighted total: 0.0
Shared items: dict_keys(['Get Shorty (1995)'])

Similarity with 4: 0
Item: Four Rooms (1995), Rating by user 4: 4.0,  Similarity sum: 0, Weighted total: 0.0
Item: Copycat (1995), Rating by user 4: 4.0,  Similarity sum: 0, Weighted total: 0.0
Shared items: dict_keys(['GoldenEye (1995)', 'Get Shorty (1995)'])

Similarity with 5: 0
Item: Toy Story (1995), Rating by user 5: 3.0,  Similarity sum: 0, Weighted total: 0.0
Item: Copycat (1995), Rating by user 5: 5.0,  Similarity sum: 0, Weighted total: 0.0


Item: Toy Story (1995), Total score: 0.0, Similarity s

In [17]:
print (recommendations)

[]
